# Step 1: Train Maps (Playground)

In [ ]:
import pandas as pd
import numpy as np
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm
import os
import shutil
from math import radians, cos

from data_lib.config import H_INSTALL, H_DECLINE, WEIGHT_DECLINE, WEIGHT_INSTALL, \
    HEX_GRID_SIZES, COMPETITION_SEARCH_RADIUS_DEG, HEX_TILING_RADIUS_KM
import data_lib.config as config
from data_lib.data_fetch.get_data import get_train_data
from data_lib.feature.spatial_weights import build_desirability_field_idw
from data_lib.geometry.hex import find_best_hexes, create_hex_grid, compute_hexes
from data_lib.geometry.find_boundary import run_find_boundary
from data_lib.test import get_overlap


## ⚠️ Bug Fix: `data_lib/geometry/hex.py`

`install_velocity` line has `config.` leaked into an f-string:
```
row[f"installs_config.{config.TEMPORAL_WINDOWS[-1]}d"]   # ← produces 'installs_config_365d'
row[f"installs_{config.TEMPORAL_WINDOWS[-1]}d"]           # ← correct: 'installs_365d'
```

The cell below patches the source file directly so ProcessPoolExecutor workers pick it up.


In [ ]:
import data_lib.geometry.hex as _hex_mod

hex_py_path = _hex_mod.__file__
print(f"Patching: {hex_py_path}")

with open(hex_py_path, "r") as f:
    src = f.read()

broken  = 'row[f"installs_config.{config.TEMPORAL_WINDOWS[-1]}d"]'
fixed   = 'row[f"installs_{config.TEMPORAL_WINDOWS[-1]}d"]'

if broken in src:
    src = src.replace(broken, fixed)
    with open(hex_py_path, "w") as f:
        f.write(src)
    print("✓ Fixed install_velocity bug in hex.py")
    
    # Force reimport so this process sees the fix too
    import importlib
    importlib.reload(_hex_mod)
    from data_lib.geometry.hex import compute_hexes
    print("✓ Module reloaded")
elif fixed in src:
    print("✓ Already fixed — nothing to do")
else:
    print("⚠️  Neither pattern found — check hex.py manually")


## 0. Setup

In [ ]:
print("--- STARTING MAP TRAINING (PLAYGROUND) ---")

ARTIFACTS_DIR = "artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

print(f"Train window: {config.TRAIN_START_DATE} -> {config.TRAIN_END_DATE}")


## 1. Get Data

In [ ]:
df_train = get_train_data(config.TRAIN_START_DATE, config.TRAIN_END_DATE)
if df_train.empty:
    raise RuntimeError("No training data. Exiting.")
print(f"Training data: {len(df_train)} rows")
df_train.head()


## 2. Build Desirability Fields

In [ ]:
print("Building Desirability Fields...")
df_train = build_desirability_field_idw(
    df_train,
    radius_meters=max(H_INSTALL, H_DECLINE),
    decline_weight=WEIGHT_DECLINE,
    install_weight=WEIGHT_INSTALL,
)

# Set 'h' based on weight sign (Custom Logic for Split H)
df_train["h"] = np.where(df_train["field_weight"] >= 0, H_INSTALL, H_DECLINE)

# Save Train Data Artifact
train_h5_path = os.path.join(ARTIFACTS_DIR, "train_data.h5")
df_train.to_hdf(train_h5_path, mode="w", key="df")
print(f"Saved {train_h5_path}")


## 3. Calculate SE Thresholds

In [ ]:
print("Generating Hexagons...")
partners = df_train["partner_id"].unique().tolist()

# Calculate SE thresholds
df_train["is_installed"] = (df_train["final_decision"] == "INSTALLED").astype(int)
df_train["is_declined"] = (df_train["final_decision"] == "DECLINED").astype(int)
df_train["is_indeterminate"] = (df_train["final_decision"] == "INDETERMINATE").astype(int)
df_train["is_hanging"] = (df_train["final_decision"] == "HANGING").astype(int)
df_train = df_train[df_train["final_decision"].isin(["INSTALLED", "DECLINED"])].copy()

dfp = df_train.groupby("partner_id").agg(
    total=("mobile", "count"), installed=("is_installed", "sum")
).reset_index()
dfp["se"] = dfp["installed"] / dfp["total"]
dfp["se_rng"] = pd.qcut(dfp["se"], q=10, labels=False, duplicates="drop") + 1

bad_se_rng = config.BAD_SE
mid_se_rng = config.MID_SE

bad_se = dfp[dfp["se_rng"].isin(bad_se_rng)]["se"].max()
mid_se = dfp[dfp["se_rng"].isin(mid_se_rng)]["se"].max()

print(f"bad_se = {bad_se:.4f}")
print(f"mid_se = {mid_se:.4f}")
print(f"Partners to process: {len(partners)}")


## 4. Hex Generation (Parallel)

In [ ]:
def process_single_partner(partner_id, df_train, bad_se, mid_se):
    """
    Worker function to process one partner's hexagons.
    """
    sub_df = df_train[df_train["partner_id"] == partner_id].copy()

    if len(sub_df) < 5:  # Min samples
        return []

    center_lat = np.median(sub_df["latitude"])
    center_lon = np.median(sub_df["longitude"])

    # Determine best hex size using configured search sizes and radius
    best_size = find_best_hexes(
        center_lat, center_lon,
        sub_df,
        hex_sizes=HEX_GRID_SIZES,
        radius_km=HEX_TILING_RADIUS_KM,
    )

    # Generate Grid using configured tiling radius
    hexes = create_hex_grid(
        center_lat,
        center_lon,
        radius_km=HEX_TILING_RADIUS_KM,
        hex_size_km=best_size,
    )

    # Compute Stats — NOW WITH reference_date for temporal buckets
    # (uses the patched compute_hexes from disk fix above)
    from data_lib.geometry.hex import compute_hexes as _compute_hexes
    hex_stats = _compute_hexes(
        hexes,
        center_lat,
        center_lon,
        sub_df,
        bad_se,
        mid_se,
        best_size,
        partner_id,
        reference_date=config.TRAIN_END_DATE,
    )

    return hex_stats


In [ ]:
hexagons = []
with ProcessPoolExecutor(max_workers=mp.cpu_count()) as executor:
    futures = {
        executor.submit(process_single_partner, pid, df_train, bad_se, mid_se): pid
        for pid in partners
    }
    for future in tqdm(as_completed(futures), total=len(futures), desc="Partners"):
        hexagons.extend(future.result())

# LET DICTS DEFINE SCHEMA — no hardcoded columns list
df_hex = pd.DataFrame(hexagons)

# Verify temporal columns made it through
temporal_check = [f"se_{wd}d" for wd in config.TEMPORAL_WINDOWS]
present = [c for c in temporal_check if c in df_hex.columns]
missing_t = [c for c in temporal_check if c not in df_hex.columns]
print(f"[HEX] Temporal columns present: {present}")
if missing_t:
    print(f"[HEX] WARNING: Temporal columns missing: {missing_t}")

# Verify install_velocity is clean
if "install_velocity" in df_hex.columns:
    n_valid = df_hex["install_velocity"].notna().sum()
    print(f"[HEX] install_velocity: {n_valid}/{len(df_hex)} valid")

# Check no ghost 'config' columns leaked in
bad_cols = [c for c in df_hex.columns if "config" in c.lower()]
if bad_cols:
    raise RuntimeError(f"Bug still present! Ghost columns: {bad_cols}")
else:
    print("[HEX] ✓ No 'config' ghost columns")

# Temporary save for find_boundary
temp_poly_path = "artifacts/poly_stats.h5"
df_hex.to_hdf(temp_poly_path, mode="w", key="df")
print(f"Saved intermediate {temp_poly_path} ({len(df_hex)} hexes, {len(df_hex.columns)} cols)")


## 5. Find Boundaries

In [ ]:
print("Finding Boundaries...")
run_find_boundary()

# Move result to artifacts if it ended up in cwd
bound_path = "partner_cluster_boundaries.h5"
artifact_bound_path = os.path.join(ARTIFACTS_DIR, "partner_cluster_boundaries.h5")

if os.path.exists(bound_path) and os.path.abspath(bound_path) != os.path.abspath(artifact_bound_path):
    shutil.move(bound_path, artifact_bound_path)
    print(f"Moved {bound_path} → {artifact_bound_path}")
else:
    print(f"Boundary file at {artifact_bound_path}")


## 6. Competition / Overlaps (Final Map)

In [ ]:
print("Computing Competition Overlaps...")

# get_overlap() reads from artifacts/ directly, but also needs
# partner_cluster_boundaries.h5 accessible in cwd — copy if needed
bound_path = "partner_cluster_boundaries.h5"
artifact_bound_path = os.path.join(ARTIFACTS_DIR, "partner_cluster_boundaries.h5")

if os.path.exists(artifact_bound_path) and not os.path.exists(bound_path):
    shutil.copy(artifact_bound_path, bound_path)

get_overlap(
    search_radius_deg=COMPETITION_SEARCH_RADIUS_DEG
)

# Move final poly output to artifacts
final_poly_path = "poly_stats_final.h5"
artifact_final_poly_path = os.path.join(ARTIFACTS_DIR, "poly_stats_final.h5")

if os.path.exists(final_poly_path) and os.path.abspath(final_poly_path) != os.path.abspath(artifact_final_poly_path):
    shutil.move(final_poly_path, artifact_final_poly_path)
    print(f"Saved {artifact_final_poly_path}")
else:
    print(f"Final poly file at {artifact_final_poly_path}")

print("--- MAP TRAINING COMPLETE ---")


## 7. Verify Outputs

In [ ]:
expected_files = [
    "artifacts/train_data.h5",
    "artifacts/poly_stats.h5",
    "artifacts/partner_cluster_boundaries.h5",
    "artifacts/poly_stats_final.h5",
]

print("Output files:")
for f in expected_files:
    exists = os.path.exists(f)
    size = os.path.getsize(f) / 1024 / 1024 if exists else 0
    status = f"✓ ({size:.1f} MB)" if exists else "✗ MISSING"
    print(f"  {f}: {status}")

# Quick column check on final poly
df_final_check = pd.read_hdf("artifacts/poly_stats_final.h5", "df")
print(f"\nFinal poly: {len(df_final_check)} rows, {len(df_final_check.columns)} cols")
print(f"Columns: {sorted(df_final_check.columns.tolist())}")

# Confirm no 'installs_config_*' ghost columns
bad_cols = [c for c in df_final_check.columns if "config" in c.lower()]
if bad_cols:
    print(f"\n⚠️ WARNING: Found 'config' in column names (bug residue): {bad_cols}")
else:
    print("\n✓ No 'config' ghost columns — bug fix confirmed clean")
